# 01 · Perfil de los datos de Lojagas

**Fase CRISP-DM:** comprensión de los datos (descripción y verificación de calidad).

**Objetivo:** confirmar el rango temporal, la granularidad, la unicidad de `linea_id`, los tipos de documento, los candidatos a segmento (I-04) y el volumen de GLP por mes en ventas y compras.

**Regla de privacidad:** este notebook no muestra filas. Solo conteos, sumas y categorías. No usar `head()`, `sample()` ni imprimir registros (C-08). Se leen únicamente las columnas de la lista permitida.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import pyarrow as pa
import pyarrow.csv as pv
from IPython.display import display

ENTRADA = Path(os.environ["LOJAGAS_DATOS_ENTRADA"]).expanduser()
VENTAS = ENTRADA / "ventas_2026-11-13.csv"
COMPRAS = ENTRADA / "compras_2026-11-13.csv"
assert VENTAS.exists() and COMPRAS.exists()
pd.set_option("display.max_rows", 100)
# Una columna con más valores distintos no se muestra: podría listar clientes (PII).
MAX_CATEGORIAS = 50


def pocas_categorias(df, col):
    n = df[col].nunique(dropna=False)
    if n > MAX_CATEGORIAS:
        print(f"[{col}] {n} valores distintos: no se muestra (posible PII)")
    return n <= MAX_CATEGORIAS

## Lectura

Lista permitida de columnas: nada de nombres, RUC, coordenadas, conductores, placas ni pagos. `newlines_in_values=True` porque hay campos de texto con saltos de línea (por ejemplo `guia_origen`).

Los errores de lectura de pyarrow incluyen el texto de la fila (con columnas prohibidas). Por eso las filas malformadas se cuentan y se saltan sin guardar su contenido, y cualquier otro error se relanza sin detalle.

In [ ]:
NUMERICAS = {c: pa.float64() for c in ["cantidad", "kg_glp", "peso_unitario_kg"]}
FECHAS = {c: pa.date32() for c in ["fecha_factura", "fecha_contable"]}
IDS = {c: pa.int64() for c in ["factura_id", "linea_id"]}


FILAS_SALTADAS = {}


def leer(ruta, columnas):
    saltadas = 0

    def saltar(_fila):
        # No se guarda ni se imprime _fila.text: trae todas las columnas, incluidas las prohibidas.
        nonlocal saltadas
        saltadas += 1
        return "skip"

    try:
        tabla = pv.read_csv(
            ruta,
            parse_options=pv.ParseOptions(newlines_in_values=True, invalid_row_handler=saltar),
            convert_options=pv.ConvertOptions(
                include_columns=columnas,
                # Sin esto, "NULL" queda como texto en las columnas de texto
                strings_can_be_null=True,
                null_values=["NULL", ""],
                column_types={**NUMERICAS, **FECHAS, **IDS},
            ),
        )
    except pa.ArrowException as e:
        # El mensaje original puede incluir valores del CSV; solo se informa el tipo de error.
        raise RuntimeError(
            f"No se pudo leer {ruta.name}: {type(e).__name__} (detalle suprimido)"
        ) from None
    FILAS_SALTADAS[ruta.name] = saltadas
    return tabla.to_pandas(date_as_object=False)


COLS_VENTAS = [
    "factura_id",
    "linea_id",
    "fecha_factura",
    "fecha_contable",
    "move_type",
    "state",
    "categoria",
    "cliente_tipo",
    "tipo_distribuidor",
    "canal_venta",
    "es_subsidiado",
    "jerarquia_cliente",
    "producto_codigo",
    "producto_nombre",
    "unidad_medida",
    "peso_unitario_kg",
    "cantidad",
    "kg_glp",
    "canton",
    "sucursal",
    "cliente_id",
]
COLS_COMPRAS = [
    "factura_id",
    "linea_id",
    "fecha_factura",
    "fecha_contable",
    "move_type",
    "state",
    "categoria",
    "producto_codigo",
    "producto_nombre",
    "peso_unitario_kg",
    "cantidad",
    "kg_glp",
    "canal_compra",
    "proveedor_id",
]

ventas = leer(VENTAS, COLS_VENTAS)
compras = leer(COMPRAS, COLS_COMPRAS)
print(f"ventas: {ventas.shape} | memoria {ventas.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(f"compras: {compras.shape}")
print(f"filas malformadas saltadas: {FILAS_SALTADAS}")

## Verificaciones básicas

In [ ]:
def resumen(df, nombre):
    meses = df["fecha_factura"].dt.to_period("M")
    return pd.Series(
        {
            "filas": len(df),
            "linea_id únicos": df["linea_id"].nunique(),
            "linea_id duplicados": int(df["linea_id"].duplicated().sum()),
            "facturas": df["factura_id"].nunique(),
            "fecha mínima": df["fecha_factura"].min(),
            "fecha máxima": df["fecha_factura"].max(),
            "meses distintos": meses.nunique(),
            "kg_glp nulos (%)": round(df["kg_glp"].isna().mean() * 100, 2),
        },
        name=nombre,
    )


display(pd.concat([resumen(ventas, "ventas"), resumen(compras, "compras")], axis=1))

**Si hay `linea_id` duplicados**, la exportación multiplicó líneas al unir despachos, turnos o guías. En ese caso no se puede sumar `kg_glp` sin deduplicar antes.

Desde aquí `ventas` y `compras` quedan con una fila por `linea_id`. Si las copias de una línea tienen kg distintos, el notebook se detiene: no se sabe cuál es la buena.

In [ ]:
def deduplicar(df, nombre):
    dup = df[df["linea_id"].duplicated(keep=False)]
    print(f"{nombre}: líneas duplicadas {dup['linea_id'].nunique()} | filas afectadas {len(dup)}")
    if len(dup):
        # Líneas cuyas copias tienen más de un valor de kg (NaN cuenta como valor)
        kg_por_linea = dup.drop_duplicates(["linea_id", "kg_glp"])["linea_id"].value_counts()
        distintos = int((kg_por_linea > 1).sum())
        if distintos:
            raise ValueError(
                f"{nombre}: {distintos} líneas con kg distintos entre copias; "
                "revisar antes de sumar"
            )
    return df.drop_duplicates("linea_id")


ventas = deduplicar(ventas, "ventas")
compras = deduplicar(compras, "compras")

## Tipos de documento y estado

In [ ]:
for nombre, df in [("ventas", ventas), ("compras", compras)]:
    t = df.groupby(["move_type", "state"], dropna=False).agg(
        filas=("linea_id", "size"), kg=("kg_glp", "sum")
    )
    print(nombre)
    display(t)

Las notas de crédito (`out_refund`) en Odoo guardan la cantidad en positivo. Si se incluyen, hay que restarlas. Decisión pendiente: restarlas o excluirlas, y documentarla.

Peso de las notas de crédito por mes y por categoría (solo documentos publicados), para decidir si se restan o se excluyen.

In [ ]:
pub = ventas[ventas["state"] == "posted"].copy()
pub["mes"] = pub["fecha_factura"].dt.to_period("M")


def peso_nc(indice):
    t = pub.pivot_table(index=indice, columns="move_type", values="kg_glp", aggfunc="sum")
    t = t.reindex(columns=["out_invoice", "out_refund"], fill_value=0).fillna(0) / 1000
    t["neto"] = t["out_invoice"] - t["out_refund"]
    t["nc_%"] = (t["out_refund"] / t["out_invoice"] * 100).round(2)
    return t.round(1)


display(peso_nc("mes"))  # toneladas
if pocas_categorias(ventas, "categoria"):
    display(peso_nc("categoria"))

## Candidatos a segmento (I-04)

In [ ]:
for col in [
    "categoria",
    "cliente_tipo",
    "tipo_distribuidor",
    "canal_venta",
    "es_subsidiado",
    "jerarquia_cliente",
    "unidad_medida",
]:
    if not pocas_categorias(ventas, col):
        continue
    t = (
        ventas.groupby(col, dropna=False)
        .agg(filas=("linea_id", "size"), kg=("kg_glp", "sum"))
        .sort_values("kg", ascending=False)
    )
    t["kg_%"] = (t["kg"] / t["kg"].sum() * 100).round(2)
    print(f"[{col}]")
    display(t)

In [ ]:
# ¿Cómo se cruzan la categoría de producto y el tipo de cliente? (kg)
if pocas_categorias(ventas, "categoria") and pocas_categorias(ventas, "cliente_tipo"):
    display(
        pd.crosstab(
            ventas["categoria"],
            ventas["cliente_tipo"],
            values=ventas["kg_glp"],
            aggfunc="sum",
            dropna=False,
        ).round(0)
    )

In [ ]:
# Productos y su peso unitario: verifica que kg_glp = cantidad × peso_unitario_kg
p = ventas.groupby(["categoria", "producto_codigo", "producto_nombre"], dropna=False).agg(
    filas=("linea_id", "size"),
    peso=("peso_unitario_kg", "median"),
    pesos_distintos=("peso_unitario_kg", "nunique"),
    kg=("kg_glp", "sum"),
)
p = p[p["kg"] > 0]
print(f"productos de GLP con más de un peso unitario: {int((p['pesos_distintos'] > 1).sum())}")
if len(p) > MAX_CATEGORIAS:
    print(f"{len(p)} productos con kg de GLP: no se muestran (posible PII)")
else:
    display(p.sort_values("kg", ascending=False))
desvio = (ventas["kg_glp"] - ventas["cantidad"] * ventas["peso_unitario_kg"]).abs() > 0.01
print(f"filas donde kg_glp ≠ cantidad × peso: {int(desvio.sum())}")

## Serie mensual de ventas

In [ ]:
v = ventas[(ventas["move_type"] == "out_invoice") & (ventas["state"] == "posted")].copy()
v["mes"] = v["fecha_factura"].dt.to_period("M")
mensual = v.pivot_table(index="mes", columns="categoria", values="kg_glp", aggfunc="sum").fillna(0)
display((mensual / 1000).round(1))  # toneladas

ax = (mensual / 1000).plot(figsize=(12, 5), marker="o")
ax.set_ylabel("toneladas")
ax.set_title("Ventas de GLP por mes y categoría (solo out_invoice publicadas)")
plt.tight_layout()
plt.show()

In [ ]:
# Meses faltantes y clientes activos por mes
rango = pd.period_range(v["mes"].min(), v["mes"].max(), freq="M")
print("meses sin ventas:", [str(m) for m in rango.difference(v["mes"].unique())])
display(v.groupby("mes")["cliente_id"].nunique().rename("clientes_activos").to_frame().T)

**Ojo con el último mes:** si la exportación se hizo a mitad de mes, ese mes está incompleto y no debe entrar en el entrenamiento ni en la evaluación.

In [ ]:
kg_mes = v.groupby("mes")["kg_glp"].sum()
kg_mes = kg_mes.reindex(
    pd.period_range(kg_mes.index.min(), kg_mes.index.max(), freq="M"), fill_value=0
)
primero, ultimo = kg_mes.index.min(), kg_mes.index.max()
print(f"primer mes {primero}: desde el día {v.loc[v['mes'] == primero, 'fecha_factura'].min().day}")
mediana_12 = kg_mes.loc[ultimo - 12 : ultimo - 1].median()
print(
    f"último mes {ultimo}: hasta el día {v.loc[v['mes'] == ultimo, 'fecha_factura'].max().day} "
    f"de {ultimo.days_in_month} | kg = {kg_mes[ultimo] / mediana_12 * 100:.0f} % "
    "de la mediana de los 12 meses previos"
)

# Huecos a mitad de la serie: meses bajo el 50 % de la mediana móvil de 12 meses (sin el último mes)
completos = kg_mes.iloc[:-1]
referencia = completos.rolling(12, min_periods=6, center=True).median()
bajos = completos[completos < 0.5 * referencia]
print(f"meses bajo el 50 % de la mediana móvil: {len(bajos)}")
if len(bajos):
    display(
        pd.DataFrame(
            {"toneladas": bajos / 1000, "referencia_t": referencia[bajos.index] / 1000}
        ).round(1)
    )

## Meses por segmento

Particiones walk-forward con horizonte 2 (datos hasta M-1 para predecir M+1): `meses − ventana − 1`. Se excluye el último mes (incompleto). Determina si la LSTM se justifica y cuántas particiones tiene la prueba pareada.

In [ ]:
ultimo_completo = v["mes"].max() - 1

for col in ["categoria", "cliente_tipo", "canal_venta"]:
    if not pocas_categorias(v, col):
        continue
    kg = (
        v[v["mes"] <= ultimo_completo]
        .pivot_table(index=col, columns="mes", values="kg_glp", aggfunc="sum", dropna=False)
        .fillna(0)
    )
    filas = []
    for valor, serie in kg.iterrows():
        activos = serie[serie > 0]
        if activos.empty:
            continue
        n = (ultimo_completo - activos.index.min()).n + 1
        filas.append(
            {
                col: valor,
                "primer mes": activos.index.min(),
                "último mes": activos.index.max(),
                "meses": n,
                "meses en cero": n - len(activos),
                **{f"particiones_{w}": max(n - w - 1, 0) for w in (12, 24, 36)},
            }
        )
    print(f"[{col}]")
    display(pd.DataFrame(filas).sort_values("particiones_24"))

## Retraso de registro (ADR-0014: día 2 o día 5)

In [ ]:
for nombre, df in [("ventas", ventas), ("compras", compras)]:
    d = (df["fecha_contable"] - df["fecha_factura"]).dt.days
    print(nombre)
    display(d.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(1).to_frame().T)
    # ¿Qué fecha define el mes? Solo importa si caen en meses distintos.
    ambas = df["fecha_factura"].notna() & df["fecha_contable"].notna()
    otro_mes = ambas & (
        df["fecha_factura"].dt.to_period("M") != df["fecha_contable"].dt.to_period("M")
    )
    print(
        f"líneas con mes contable ≠ mes de factura: {int(otro_mes.sum())} "
        f"({otro_mes.mean() * 100:.2f} % de filas, "
        f"{df.loc[otro_mes, 'kg_glp'].sum() / df['kg_glp'].sum() * 100:.2f} % de kg)"
    )

Esto mide la diferencia entre la fecha de la factura y la contable, no la fecha real de registro en Odoo (`create_date` no viene en la exportación). Si hace falta, pedirla a Lojagas.

Aproximación por el orden de creación: Odoo asigna `factura_id` en secuencia al crear el documento. Una factura del mes M con id mayor que las facturas fechadas desde el día k de M+1 se creó después de ese día. Como umbral se usa el percentil 1 de esos id, para que unas pocas facturas con fecha futura no lo muevan.

Subestima el retraso: el id se asigna al crear el borrador, no al publicarlo. Para cerrar el pendiente hace falta `create_date` o la fecha de publicación.

In [ ]:
f = (
    v.groupby("factura_id")
    .agg(fecha=("fecha_factura", "first"), mes=("mes", "first"), kg=("kg_glp", "sum"))
    .reset_index()
)
filas = []
for mes in sorted(f["mes"].unique())[:-1]:  # el último mes no tiene M+1
    del_mes = f[f["mes"] == mes]
    sig = f[f["mes"] == mes + 1]
    fila = {"mes": mes}
    for k in (2, 5):
        umbral = sig.loc[sig["fecha"].dt.day >= k, "factura_id"].quantile(0.01)
        fila[f"kg_tras_dia_{k}_%"] = (
            del_mes.loc[del_mes["factura_id"] > umbral, "kg"].sum() / del_mes["kg"].sum() * 100
        )
    filas.append(fila)
retraso = pd.DataFrame(filas).set_index("mes").round(2)
display(retraso.describe(percentiles=[0.5, 0.9]).round(2))
display(retraso)

## Compras: solo GLP

In [ ]:
if pocas_categorias(compras, "categoria"):
    t = compras.groupby(["categoria"], dropna=False).agg(
        filas=("linea_id", "size"), kg=("kg_glp", "sum"), con_kg=("kg_glp", "count")
    )
    display(t.sort_values("kg", ascending=False))

In [ ]:
c = compras[
    compras["kg_glp"].notna()
    & (compras["state"] == "posted")
    & (compras["move_type"] == "in_invoice")
].copy()
c["mes"] = c["fecha_factura"].dt.to_period("M")
comp = pd.DataFrame(
    {
        "ventas_t": v.groupby("mes")["kg_glp"].sum() / 1000,
        "compras_t": c.groupby("mes")["kg_glp"].sum() / 1000,
    }
)
comp["ventas/compras"] = (comp["ventas_t"] / comp["compras_t"]).round(3)
display(comp.round({"ventas_t": 1, "compras_t": 1}))

ax = comp[["ventas_t", "compras_t"]].plot(figsize=(12, 5), marker="o")
ax.set_ylabel("toneladas")
ax.set_title("Ventas frente a compras de GLP por mes")
plt.tight_layout()
plt.show()

In [ ]:
# Inventario implícito: compras acumuladas − ventas acumuladas. El nivel no es real
# (falta el inventario inicial y las pérdidas); importa la forma.
UMBRAL_RATIO = 0.97
inventario = (comp["compras_t"].fillna(0) - comp["ventas_t"].fillna(0)).cumsum()
# Posible cupo: se vendió casi todo lo comprado (ventas ≈ compras). Un tramo plano del
# inventario lo confirma; uno que sube indica que quedó gas sin vender.
comp["inventario_implicito_t"] = inventario
comp["posible_cupo"] = comp["ventas/compras"] >= UMBRAL_RATIO
print(f"meses con posible cupo: {int(comp['posible_cupo'].sum())} de {len(comp)}")
display(comp.round({"ventas_t": 1, "compras_t": 1, "inventario_implicito_t": 1}))

ax = inventario.plot(figsize=(12, 4), marker="o")
ax.set_ylabel("toneladas")
ax.set_title("Inventario implícito (compras − ventas acumuladas)")
plt.tight_layout()
plt.show()

Meses con `ventas/compras` cercano a 1 y sin inventario sobrante sugieren que la venta estuvo limitada por el cupo: la demanda real pudo ser mayor que lo vendido. Esos meses se marcan antes de modelar.

## Hallazgos

_(Completar al ejecutar: rango, duplicados, decisión sobre notas de crédito, campo de segmento elegido, particiones por segmento, meses incompletos, meses con posible cupo, retraso de registro.)_